# Web scraping IMdB for Quentin Tarantino ratings and film context

In [132]:
#Installing required libraries for webscraping
!pip install requests beautifulsoup4 pandas

In [133]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd

# IMDb page specific to Quentin Tarantino
url = 'https://www.imdb.com/list/ls069398589/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'
}

# Fetching page
response = requests.get(url, headers =headers)
soup = BeautifulSoup(response.content, 'html.parser')

# Finding data 
json_data = soup.find('script', type='application/ld+json')


# Get JSON data
data = json.loads(json_data.string)

# Extract movie list from the 'itemListElement'
movies = []
for i, entry in enumerate(data['itemListElement'], start = 1):
    item = entry['item']
    movies.append({
        'IMDb Ranking': i,
        'Title': item.get('name'),
        'Rating': item['aggregateRating']['ratingValue'] if 'aggregateRating' in item else None,
        'Votes': item['aggregateRating']['ratingCount'] if 'aggregateRating' in item else None,
        'Genre': item.get('genre'),
        'Duration': item.get('duration'),
        'Content Rating': item.get('contentRating'),
        'URL': item.get('url'),
        'Description': item.get('description'),
        'Poster': item.get('image')
    })

# Making data frame
QT_movies_df = pd.DataFrame(movies)

# Save the DataFrame to a CSV file
QT_movies_df.to_csv('Quentin_Tarantino_Movies.csv', index=False)


Cleaning the csv file

In [134]:
import pandas as pd
import re

#Loading initial webscraped data set to then clean.
QT_movies_df = pd.read_csv('Quentin_Tarantino_Movies.csv')

# removing the specific movies- he has directoral credits beyond the 10 movies of interest, those are unwanted.
QT_movies_df = QT_movies_df[~QT_movies_df['Title'].isin(["My Best Friend&apos;s Birthday", "Four Rooms"])]

# Changing duration column into minutes.
def clean_duration(duration_str):
    if isinstance(duration_str, str):
        
# Matching the original pattern.
        time_match =re.match(r'PT(\d+)H(\d+)M', duration_str)
        if time_match:
            hours = int(time_match.group(1))
            minutes= int(time_match.group(2))
            return hours *60 +minutes
    return None

# applying the cleaned times into the column.
QT_movies_df['Duration (Minutes)'] =QT_movies_df['Duration'].apply(clean_duration)

# Making sure the original Duration column is gone
QT_movies_df.drop(columns=['Duration'], inplace=True)

columns = ['IMDb Ranking','Title', 'Duration (Minutes)','Rating', 'Votes', 'Genre','Content Rating', 'URL', 'Description', 'Poster']
QT_movies_df = QT_movies_df[columns]

# saving new data frame and making new csv data with the cleaned data.
QT_movies_df.to_csv('Quentin_Tarantino_IMDB_Cleaned.csv', index=False)


# Web scraping to get kill data on the films (uncleaned data sets)

Reservoir Dogs (1992)

In [135]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the Reservoir Dogs death list
url1 = "https://listofdeaths.fandom.com/wiki/Reservoir_Dogs"

#requesting 
response =requests.get(url1)
soup = BeautifulSoup(response.content,"html.parser")

content= soup.find("div", class_="mw-parser-output")
deaths = []

# Predefined gender mapping to help data cleaning easier down the line.
gender_map = {
    "Unnamed Woman": "Female",
    "Unnamed Police Officer": "Male",
    "Unnamed Police Officers": "Male",
    "Officer Marvin Nash": "Male",
    "Officer Frederick \"Freddy\" Newandyke/Mr. Orange": "Male",
    "Victor \"Vic\" Vega/Mr. Blonde": "Male",
    "Joseph \"Joe\" Cabot": "Male",
    "Edward \"Nice Guy Eddie\" Cabot": "Male",
    "Lawrence \"Larry\" Dimmick/Mr. White": "Male",
    "\"Mr. Blue\"": "Male",
    "\"Mr. Brown\"": "Male",
    "\"Mr. Pink\"": "Male",
}

# looping through all bullet points.
for li in content.find_all("li"):
    text= li.get_text().strip()
    
#stop if we hit kill summaries to help make cleaning easier later.
    if "Total" in text or "Victor" in text or "Lawrence" in text or "Officer Frederick" in text or "\"Mr. Pink\"" in text:
        break

# Split into 'Who Died' and 'Description'
    if ' - ' in text:
        who, desc= text.split(' - ', 1)
        who = who.strip()
        desc =desc.strip()

# Making sure aggregates are counted as indivoduals.
        if who.startswith("Four Unnamed Clerks"):
            for i in range(1, 5):
                deaths.append({
                    "Who Died": f"Unnamed Clerk #{i}",
                    "Description": desc,
                    "Gender": "Male"
                })
        elif who.startswith("Two Unnamed Police Officers"):
            for i in range(1, 3):
                deaths.append({
                    "Who Died": f"Unnamed Police Officer #{i}",
                    "Description": desc,
                    "Gender": "Male"
                })
        else:
# try matching, fall back to "Unknown", however this will be edited further down the line
            matched_gender ="Unknown"
            for key in gender_map:
                if who.startswith(key) or key in who:
                    matched_gender = gender_map[key]
                    break
            deaths.append({
                "Who Died": who,
                "Description": desc,
                "Gender": matched_gender
            })

df_resdog_death_uncleaned = pd.DataFrame(deaths)
#Saving to csv as want records of all files
df_resdog_death_uncleaned.to_csv("reservoir_dogs_deaths_uncleaned.csv", index=False)



Pulp Fiction (1994)

In [136]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

#URL of the Pulp Fiction death list
url2 = "https://listofdeaths.fandom.com/wiki/Pulp_Fiction"

#Requesting the URL
response= requests.get(url2)
soup =BeautifulSoup(response.content, "html.parser")

content= soup.find("div", class_="mw-parser-output")
deaths = []

#Creating gender map to make cleaning easier down the line.
gender_map = {
    "Roger": "Male",
    "Brett": "Male",
    "Unnamed Man": "Male",
    "Marvin": "Male",
    "Conrad": "Male",
    "Ginny": "Female",
    "Mia Wallace": "Female",
    "Butch Coolidge's Father": "Male",
    "Dane Coolidge": "Male",
    "Floyd Wilson": "Male",
    "Vincent Vega": "Male",
    "Unnamed Woman": "Female",
    '"The Gimp"': "Male",
    "Maynard": "Male",
    'Zedekiah "Zed"': "Male"
}

#looping through all points
for li in content.find_all("li"):
    text =li.get_text().strip()

# Skipping irrelevant data.
    if "Total" in text or "Portrayed" in text or "Deaths caused by" in text:
        continue
    
#Only including data entries with correct format.
    if ' - ' in text:
        who, desc= text.split(' - ', 1)
        who = who.strip()
        desc =desc.strip()

#Using unknown if data doesn't match, but this will be ammended down the line.
        matched_gender = "Unknown"
        for key in gender_map:
            if who.startswith(key) or key in who:
                matched_gender= gender_map[key]
                break

        deaths.append({
            "Who Died": who,
            "Description": desc,
            "Gender": matched_gender
        })

#Creating data frame for join later
df_pulp_fiction_death_uncleaned = pd.DataFrame(deaths)

#Saving to csv as want records of all files to check
df_pulp_fiction_death_uncleaned.to_csv("pulp_fiction_deaths_uncleaned.csv", index=False)



Jackie Brown (1997)

In [137]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the Jackie Brown death list
url3 = "http://listofdeaths.fandom.com/wiki/Jackie_Brown"

#Requesting the URL
response= requests.get(url3)
soup = BeautifulSoup(response.content, "html.parser")

# Target the main content area
content = soup.find("div", class_="mw-parser-output")
deaths= []

#Creating gender map to make cleaning easier down the line.
gender_map = {
    "Beaumont Livingston": "Male",
    "Melanie Ralston": "Female",
    "Louis Gara": "Male",
    "Ordell Robbie": "Male"
}

# Looping through bullet points 
for li in content.find_all("li"):
    text =li.get_text().strip()

# Skipping unimportant data or total counts
    if "Total" in text or "Kill Counts" in text:
        continue

# Only include death entries that are in correct format.
    if ' - ' in text:
        who, desc = text.split(' - ', 1)
        who = who.strip()
        desc = desc.strip()
        
        
        if who in gender_map:
            matched_gender= gender_map[who]
            deaths.append({
                "Who Died": who,
                "Description": desc,
                "Gender": matched_gender
            })

#Creating data frame fo join later
df_jackie_brown_death_uncleaned = pd.DataFrame(deaths)

#Saving to csv as want records of all files to check
df_jackie_brown_death_uncleaned.to_csv("jackie_brown_deaths_uncleaned.csv", index=False)




Kill Bill Vol 1 (2003)

In [138]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the Kill Bill death list
url4 = "https://listofdeaths.fandom.com/wiki/Kill_Bill#Volume_One"

#Requesting the URL
response = requests.get(url4)
soup = BeautifulSoup(response.content, "html.parser")

content = soup.find("div", class_="mw-parser-output")
deaths= []

#Creating gender map to make cleaning easier down the line.
gender_map= {
    'Thomas "Tommy" Plympton': "Male",
    "Rufus": "Male",
    "Reverend Harmony": "Male",
    "Mrs. Harmony": "Female",
    "Erica": "Female",
    "Joleen": "Female",
    "Janeen": "Female",
    "Unnamed Man": "Male",
    "Jasper": "Male",
    "Buck": "Male",
    "Boss Matsumoto's Guard": "Male",
    "O-Ren Ishii's Father": "Male",
    "O-Ren Ishii's Mother": "Female",
    "Boss Matsumoto": "Male",
    "Unnamed President of Panama": "Male",
    "Boss Tanaka": "Male",
    "Miki": "Male",
    "Gogo Yubari": "Female",
    "Johnny Mo": "Male",
    "O-Ren Ishii/Cottonmouth": "Female",
    "Vernita Yvonne Green/Copperhead": "Female",

}

# Looping through bullet points.
for li in content.find_all("li"):
    text= li.get_text().strip()

# Skipping unimportant data.
    if "Total" in text or "Kill Counts" in text:
        continue

# Only including correct data in the right format.
    if ' - ' in text:
        who, desc = text.split(' - ', 1)
        who = who.strip()
        desc = desc.strip()

#Dealing with aggregate kills as we want each counted individually.
        if who.startswith("Two of Boss Matsumoto's Guards"):
            for i in range(1, 3):
                deaths.append({
                    "Who Died": f"Boss Matsumoto's Guard #{i}",
                    "Description": desc,
                    "Gender": "Unknown"
                })
        elif who.startswith("Five Unnamed Members of The Crazy 88"):
            for i in range(1, 6):
                deaths.append({
                    "Who Died": f"Unnamed Member of The Crazy 88 #{i}",
                    "Description": desc,
                    "Gender": "Unknown"
                })
        elif who.startswith("Seven Unnamed Members of The Crazy 88"):
            for i in range(1, 8):
                deaths.append({
                    "Who Died": f"Unnamed Member of The Crazy 88 #{i}",
                    "Description": desc,
                    "Gender": "Unknown"
                })
        elif "40 Unnamed Members" in who:
            for i in range(1, 41):
                deaths.append({
                    "Who Died": f"Unnamed Member of the Crazy 88 #{i}",
                    "Description": desc,
                    "Gender": "Unknown"
                })
        elif "13 Unnamed Members" in who:
            for i in range(1, 14):
                deaths.append({
                    "Who Died": f"Unnamed Member of the Crazy 88 #{i}",
                    "Description": desc,
                    "Gender": "Unknown"
                })

       
        else:
# Allowing unknown, but will be delt with later.
            matched_gender = "Unknown"
            for key in gender_map:
                if key in who:
                    matched_gender = gender_map[key]
                    break
            deaths.append({
                "Who Died": who,
                "Description": desc,
                "Gender": matched_gender
            })
    

#Creating data frame for join later
df_kill_bill1_death_uncleaned = pd.DataFrame(deaths)

#Saving to csv as want records of all files to check
df_kill_bill1_death_uncleaned.to_csv("kill_bill1_deaths_uncleaned.csv", index=False)


Kill Bill: Vol 2 (2004)

In [139]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the Kill Bill Vol 2 death list
url5= "https://listofdeaths.fandom.com/wiki/Kill_Bill:_Volume_2"

#Requesting the URL
response = requests.get(url5)
soup = BeautifulSoup(response.content, "html.parser")

# Target the main content area
content= soup.find("div", class_= "mw-parser-output")
deaths = []

#Creating gender map to make cleaning easier down the line.
gender_map= {
    "Budd/Sidewinder": "Male",
    "Pai Mei": "Male",
    "Elle Driver/California Mountain Snake": "Female",
    "Bill/Snake Charmer": "Male"
}

# Looping through bullet points.
for li in content.find_all("li"):
    text = li.get_text().strip()

# Processing lines with corect format.
    if ' - ' in text:
        who, desc = text.split(' - ', 1)
        who = who.strip()
        desc= desc.strip()

#If match is failed, use unknown, but will be delt with later.
        matched_gender = "Unknown"
        for key in gender_map:
            if key in who:
                matched_gender= gender_map[key]
                break

        deaths.append({
            "Who Died": who,
            "Description": desc,
            "Gender": matched_gender
        })

#Creating data frame fo join later
df_kill_bill2_uncleaned = pd.DataFrame(deaths)

#Saving to csv as want records of all files to check
df_kill_bill2_uncleaned.to_csv("kill_bill2_deaths_uncleaned.csv", index=False)


Death Proof (2007)

In [140]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the Death Proof death list
url6 = "https://listofdeaths.fandom.com/wiki/Death_Proof"

#Requesting the URL.
response = requests.get(url6)
soup = BeautifulSoup(response.content, "html.parser")

content = soup.find("div", class_="mw-parser-output")
deaths = []

#Creating gender map to make cleaning easier down the line.
gender_map = {
    "Stuntman Mike McKay": "Male",
    "Pam": "Female",
    "Arlene": "Female",
    "Shanna": "Female",
    "Lanna": "Female",
    "Julia": "Female"
}

# Looping through bullet points.
for li in content.find_all("li"):
    text = li.get_text().strip()

# Only process if text is in correct format.
    if ' - ' in text:
        who, desc = text.split(' - ', 1)
        who = who.strip()
        desc = desc.strip()

        matched_gender = "Unknown"
        for key in gender_map:
            if key in who:
                matched_gender = gender_map[key]
                break

        deaths.append({
            "Who Died": who,
            "Description": desc,
            "Gender": matched_gender
        })

#Creating data frame fo join later
df_death_proof_uncleaned = pd.DataFrame(deaths)

#Saving to csv as want records of all files to check
df_death_proof_uncleaned.to_csv("death_proof_deaths_uncleaned.csv", index=False)


Inglorious Basterds (2009)

In [141]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# URL of the Inglourious Basterds death list
url7 ="https://listofdeaths.fandom.com/wiki/Inglourious_Basterds"

#Requesting the URL
response = requests.get(url7)
soup = BeautifulSoup(response.content, "html.parser")

content = soup.find("div", class_= "mw-parser-output")
deaths= []

#Creating gender map to make cleaning easier down the line.
gender_map = {
    "Shosanna Dreyfus": "Female",
    "Fredrick Zoller": "Male",
    "Marcel": "Male",
    "Bridget von Hammersmark": "Female",
    "Hugo Stiglitz": "Male",
    "Donny Donowitz": "Male",
    "Aldo Raine": "Male",
    "Adolf Hitler": "Male",
    "Joseph Goebbels": "Male",
    "Martin Bormann": "Male",
    "Winston Churchill": "Male",
    "Col. Hans Landa": "Male",
    "Lt. Archie Hicox": "Male",
    "Major Dieter Hellstrom": "Male",
    "Sgt. Werner Rachtman": "Male",
    "Pvt. Butz": "Male",
    "Pvt. Willi": "Male",
    "Pfc. Hirschberg": "Male",
    "Pfc. Andy Kagan": "Male",
    "Jakob Dreyfus": "Male",
    "Miriam Dreyfus": "Female",
    "Bob Dreyfus": "Male",
    "Amos Dreyfus": "Male",
    "Eric": "Male",
    "Mathilda": "Female",
    "Francesca Mondino": "Female",
    "Nazi Sergeant Wilhelm": "Male",
    "Wilhelm Wicki": "Male",
    "Omar Ulmer": "Male",
    "Hermann": "Male",
    "Private First Class Omar Ulmer": "Male",
    "Staff Sergeant Donny Donowitz/The Bear Jew": "Male",
    "Colonel Hans Landa/The Jew Hunter": "Male",
    'Lieutenant Archibald "Archie" Hicox': "Male"
}

#Function to extract number from text, this will help with aggregates later
def extract_number(text):
    match = re.search(r'(\d+)', text)
    return int(match.group(1)) if match else 1

#Looping through bullet points
for li in content.find_all("li"):
    text = li.get_text().strip()

# Skipping irrelevant lines of data.
    if "Total" in text or "Kill Count" in text:
        continue

    if ' - ' in text:
        who, desc = text.split(' - ', 1)
        who = who.strip()
        desc = desc.strip()
        
#Dealing with aggregates.
        aggregate_match = re.match(r'(\d+)\s+(Unnamed|Unnamed Nazi|Unnamed Nazi Soldiers|Unnamed Nazis|Unnamed German Soldiers|Unnamed Members of The Basterds|Unnamed Man|Unnamed Characters)', who)
        if aggregate_match:
            count = extract_number(who)
            base_name =re.sub(r'^\d+\s+', '', who).rstrip('s')
            for i in range(1, count + 1):
                deaths.append({
                    "Who Died": f"{base_name} #{i}",
                    "Description": desc,
                    "Gender": "Male" if "Nazi" in base_name or "Soldier" in base_name else "Unknown"
                })
        else:
            matched_gender= gender_map.get(who, "Unknown")
            deaths.append({
                "Who Died": who,
                "Description": desc,
                "Gender": matched_gender
            })

#Creating data frame for join later
df_basterds_uncleaned = pd.DataFrame(deaths)

#Saving to csv as want records of all files to check
df_basterds_uncleaned.to_csv("inglourious_basterds_deaths_uncleaned.csv", index=False)


Django Unchained (2012)

In [142]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the Django Unchained death list
url8= "https://listofdeaths.fandom.com/wiki/Django_Unchained"

#Requesting the URL
response= requests.get(url8)
soup =BeautifulSoup(response.content, "html.parser")
content= soup.find("div", class_="mw-parser-output")
deaths = []

#Creating gender map to make cleaning easier down the line.
gender_map = {
    "Django": "Male",
    "Dr. King Schultz": "Male",
    "Stephen": "Male",
    "Calvin J. Candie": "Male",
    "Broomhilda von Shaft": "Female",
    "Big Daddy": "Male",
    "Ace Woody": "Male",
    "Stephen's Wife": "Female",
    "Undisclosed Man": "Male", 
    "Janie": "Female",
    "Franklin": "Male",
    "Eli": "Male",
    "Undisclosed Woman": "Female",
    "Dickey Speck": "Male",
    "Sheriff Bill Sharp/Willard Peck": "Male",
    "Old Man Carrucan": "Male",
    'Jonathan "Big Jon" Brittle': "Male",
    'Roger "Lil Raj" Brittle': "Male",
    "Ellis Brittle": "Male",
    "Spencer Gordon Bennet/Big Daddy": "Male",
    "Smitty Bacal": "Male",
    "Chuck Wilson": "Male",
    "Bobby Lowe": "Male",
    "Luigi": "Male",
    "D'Artagnan": "Male",
    "Old Ben": "Male",
    "Butch Pooch": "Male",
    "Leonide Moguy": "Male",
    "Jessie": "Female",
    "Royd": "Male",
    "Reno": "Male",
    "Frankie": "Male",
    "Mr. Stonesipher": "Male",
    "Billy Crash": "Male",
    "Lara Lee Candie-Fitzwilly": "Female",
    "Stephen": "Male"
}

# Converting words into numbers for aggregate killings.
def word_to_int(word):
    word_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5,
        "Six": 6,
        "Seven": 7,
        "Eight": 8,
        "Nine": 9,
        "Ten": 10
    }
    return word_map.get(word, 0)

#Function to process aggregates into individual deaths
def process_aggregates(who, desc):
    num= 0
    if "Unnamed Male" in who:
        words = who.split(' ')[0]  
        num= word_to_int(words)  
        for i in range(1, num + 1):
            deaths.append({
                "Who Died": f"Unnamed Man #{i}",
                "Description": desc,
                "Gender": "Male"
            })
    elif "Unnamed Female" in who:
        words = who.split(' ')[0]  
        num = word_to_int(words)  
        for i in range(1, num + 1):
            deaths.append({
                "Who Died": f"Unnamed Woman #{i}",
                "Description": desc,
                "Gender": "Female"
            })

# Looping through bullet points.
for li in content.find_all("li"):
    text = li.get_text().strip()
# Skipping unrequired data.
    if "Total" in text or "Kill Counts" in text:
        continue

#Only include death entries by ensuring they contain a dash
    if ' - ' in text:
        who, desc = text.split(' - ', 1)
        who= who.strip()
        desc =desc.strip()

# Processing the names of aggregates.
        if "Unnamed Male" in who or "Unnamed Female" in who:
            process_aggregates(who, desc)
        else:
            # Regular deaths or mapped names
            matched_gender = "Unknown"
            for key in gender_map:
                if who.startswith(key) or key in who:
                    matched_gender= gender_map[key]
                    break
            deaths.append({
                "Who Died": who,
                "Description": desc,
                "Gender": matched_gender
            })

#Creating data frame for join later
df_django_deaths_uncleaned = pd.DataFrame(deaths)

#Saving to csv as want records of all files to check
df_django_deaths_uncleaned.to_csv("django_unchained_deaths_uncleaned.csv", index=False)




The Hateful Eight (2015)

In [143]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the Reservoir Dogs death list
url9 = "https://listofdeaths.fandom.com/wiki/The_Hateful_Eight"

#Requesting the URL
response = requests.get(url9)
soup = BeautifulSoup(response.content, "html.parser")
content = soup.find("div", class_= "mw-parser-output")
deaths = []

#Creating gender map to make cleaning easier down the line.
gender_map= {
    "Betsy Smithers":"Female",
    "Chester Charles Smithers":"Male",
    "Unnamed Former Sheriff of Red Rock ":"Male",
    "Oswaldo Mobray":"Male",
    "Minnie Mink": "Female",
    "Ed":"Male",
    "Sweet Dave":"Male",
    "Gemma":"Female",
    "Six Horse Judy":"Female",
    "Charly":"Male",
    "General Sanford Smithers":"Male",
    "O.B.":"Male",
    "John Ruth/The Hangman":"Male",
    "Marco the Mexican/Señor Bob":"Male",
    "Jody Domergue ":"Female",
    "Grouch Douglas/Joe Gage ":"Male",
    "Pete Hicox/Oswaldo Mobray":"Male",
    "Daisy Domergue": "Female",
    "Sheriff Chris Mannix": "Male",
    "Major Marquis Warren ":"Male"
}

# Looping through bullet points.
for li in content.find_all("li"):
    text = li.get_text().strip()


#Getting data which is in the desired format.
    if ' - ' in text:
        who, desc = text.split(' - ', 1)
        who = who.strip()
        desc = desc.strip()

#Dealing with aggregates.
        if who.startswith("Ten Unnamed Confederate Soldiers"):
            for i in range(1, 11):
                deaths.append({
                    "Who Died": f"Unnamed Confederate Soldiers",
                    "Description": desc,
                    "Gender": "Male"
                })
        elif who.startswith("37 Unnamed Union Soldiers"):
            for i in range(1, 38):
                deaths.append({
                    "Who Died": f"Unnamed Union Soldiers ",
                    "Description": desc,
                    "Gender": "Male"
                })
                
        elif who.startswith("Three Unknown Men"):
            for i in range(1, 4):
                deaths.append({
                    "Who Died": f"Unnamed Union Soldiers ",
                    "Description": desc,
                    "Gender": "Male"
                })
        else:
# Falling back to "Unknown" if unable to gender mathc, but this will be edited later down the line.
            matched_gender= "Unknown"
            for key in gender_map:
                if who.startswith(key) or key in who:
                    matched_gender= gender_map[key]
                    break
            deaths.append({
                "Who Died": who,
                "Description": desc,
                "Gender": matched_gender
            })
        
import re

for death in deaths:
    death["Description"]= re.sub(r'Total\s*-\s*\d+', '', death["Description"]).strip()


#Creating data frame for join later
df_hateful_eight_kills_uncleaned = pd.DataFrame(deaths)

#Saving to csv as want records of all files to check.
df_hateful_eight_kills_uncleaned.to_csv("hateful_eight_kills_uncleaned.csv", index=False)


Once Upon a Time...in Hollywood (2019)

In [144]:
# URL of the Kill Bill Vol 2 death list
url10 ="https://listofdeaths.fandom.com/wiki/Once_Upon_a_Time..._in_Hollywood"

#Requesting the URL
response= requests.get(url10)
soup = BeautifulSoup(response.content, "html.parser")

content = soup.find("div", class_="mw-parser-output")
deaths = []

#Creating gender map to make cleaning easier down the line.
gender_map= {
    "Billie Booth": "Male",
    'Charles Denton "Tex" Watson Jr.': "Male",
    'Patricia Dianne "Katie" Krenwinkel': "Female",
    'Susan Denise "Sadie" Atkins': "Female"
}

#Looping through bullet points 
for li in content.find_all("li"):
    text = li.get_text().strip()

    if ' - ' in text:
        who, desc = text.split(' - ', 1)
        who = who.strip()
        desc = desc.strip()

        matched_gender = "Unknown"
        for key in gender_map:
            if key in who:
                matched_gender = gender_map[key]
                break

        deaths.append({
            "Who Died": who,
            "Description": desc,
            "Gender": matched_gender
        })

# #Creating data frame for join later
df_once_upon_deaths_uncleaned = pd.DataFrame(deaths)

#Saving to csv as want records of all files to check
df_once_upon_deaths_uncleaned.to_csv("once_upon_deaths_uncleaned.csv", index=False)


# Cleaning of webscraped data about the movies (specifically focusing on kill count per movie)

Reservoir Dogs (1992)

In [145]:
import pandas as pd

# Uploading the uncleaned CSV file into a new data frame
df_res_dog_kill= pd.read_csv("reservoir_dogs_deaths_uncleaned.csv") 

# Adding movie title column
df_res_dog_kill["Movie"] = "Reservoir Dogs (1992)"

# Moving 'Movie' column to be the first of the columns.
cols = ["Movie"] + [col for col in df_res_dog_kill.columns if col != "Movie"]
df_res_dog_kill =df_res_dog_kill[cols]

# Saving to clean csv file, to keep track 
df_res_dog_kill.to_csv("reservoir_dogs_deaths_cleaned.csv", index=False)


Pulp Fiction (1994)

In [146]:

# Uploading the uncleaned CSV file into a new data frame
df_pulp_fiction_kill= pd.read_csv("pulp_fiction_deaths_uncleaned.csv")
# Adding movie title column
df_pulp_fiction_kill["Movie"] = "Pulp Fiction (1994)"

# Moving 'Movie' column to be the first of the columns.
cols = ["Movie"] + [col for col in df_pulp_fiction_kill.columns if col != "Movie"]
df_pulp_fiction_kill =df_pulp_fiction_kill[cols]

#Removing unneccesary rows that have been unnecesarily webscraped
df_pulp_fiction_kill =df_pulp_fiction_kill[:-4]

# Saving to clean csv file, to keep track
df_pulp_fiction_kill.to_csv("pulp_fiction_deaths_cleaned.csv", index = False)



Jackie Brown (1997)

In [147]:
# Uploading the uncleaned CSV file into a new data frame
df_jackie_brown_kill= pd.read_csv("jackie_brown_deaths_uncleaned.csv")
# Adding movie title column.
df_jackie_brown_kill["Movie"] = "Jackie Brown (1997)"

# Moving 'Movie' column to be the first of the columns.
cols = ["Movie"] + [col for col in df_jackie_brown_kill.columns if col != "Movie"]
df_jackie_brown_kill =df_jackie_brown_kill[cols]

#Removing unneccesary rows that have been unnecesarily webscraped
df_jackie_brown_kill = df_jackie_brown_kill[:-2]

# Saving to clean csv file, to keep track.
df_jackie_brown_kill.to_csv("jackie_brown_deaths_cleaned.csv", index = False)



Kill Bill: Volume 1 (2003)

In [148]:
# Uploading the uncleaned CSV file into a new data frame.
df_kill_bill1_kill = pd.read_csv("kill_bill1_deaths_uncleaned.csv")
# Adding movie title column
df_kill_bill1_kill["Movie"] = "Kill Bill: Volume 1 (2003)"

# Moving Movie column to be the first of the columns.
cols= ["Movie"] + [col for col in df_kill_bill1_kill.columns if col != "Movie"]
df_kill_bill1_kill = df_kill_bill1_kill[cols]

#Removing unneccesary rows that have been unnecesarily webscraped
df_kill_bill1_kill= df_kill_bill1_kill[:-13]
# Saving to clean csv file, to keep track
df_kill_bill1_kill.to_csv("kill_bill1_deaths_cleaned.csv", index = False)



Kill Bill: Volume 2 (2004)

In [149]:
#Uploading the uncleaned CSV file into a new data frame
df_kill_bill2_kill = pd.read_csv("kill_bill2_deaths_uncleaned.csv")

# Adding movie title column
df_kill_bill2_kill["Movie"] = "Kill Bill: Volume 2 (2004)"

# Moving 'Movie' column to be the first of the columns.
cols = ["Movie"] + [col for col in df_kill_bill2_kill.columns if col != "Movie"]
df_kill_bill2_kill = df_kill_bill2_kill[cols]
#Removing unneccesary rows that have been unnecesarily webscraped
df_kill_bill2_kill =df_kill_bill2_kill[:-4]
# Saving to clean csv file, to keep track
df_kill_bill2_kill.to_csv("kill_bill2_deaths_cleaned.csv", index = False)


Death Proof (2007)

In [150]:
# Uploading the uncleaned CSV file into a new data frame
df_death_proof_kill = pd.read_csv("death_proof_deaths_uncleaned.csv")
# Adding movie title column
df_death_proof_kill["Movie"] = "Death Proof (2007)"

# Moving 'Movie' column to be the first of the columns
cols = ["Movie"] + [col for col in df_death_proof_kill.columns if col != "Movie"]
df_death_proof_kill = df_death_proof_kill[cols]
#Removing unneccesary rows that have been unnecesarily webscraped
df_death_proof_kill= df_death_proof_kill[:-5]
# Saving to clean csv file, to keep track
df_death_proof_kill.to_csv("death_proof_deaths_cleaned.csv", index = False)



Inglourious Basterds (2009)

In [151]:
# Uploading the uncleaned CSV file into a new data frame.
df_ing_bast_kills = pd.read_csv("inglourious_basterds_deaths_uncleaned.csv")

# Adding movie title column.
df_ing_bast_kills["Movie"] ="Inglourious Basterds (2009)"

#Moving 'Movie' column to be the first of the columns.
cols= ["Movie"] + [col for col in df_ing_bast_kills.columns if col != "Movie"]
df_ing_bast_kills = df_ing_bast_kills[cols]
#Removing unneccesary rows that have been unnecesarily webscraped
df_ing_bast_kills = df_ing_bast_kills[:-8]
# Saving to clean csv file, to keep track
df_ing_bast_kills.to_csv("inglourious_basterds_deaths_cleaned.csv", index = False)


Django Unchained (2012)

In [152]:
#Uploading the uncleaned CSV file into a new data frame
df_django_unch_kill = pd.read_csv("django_unchained_deaths_uncleaned.csv")

# Adding movie title column
df_django_unch_kill["Movie"] = "Django Unchained (2012)"

#Moving 'Movie' column to be the first of the columns
cols= ["Movie"] + [col for col in df_django_unch_kill.columns if col != "Movie"]
df_django_unch_kill = df_django_unch_kill[cols]
#Removing unneccesary rows that have been unnecesarily webscraped
df_django_unch_kill= df_django_unch_kill[:-9]

# Saving to clean csv file, to keep track
df_django_unch_kill.to_csv("django_unchained_deaths_cleaned.csv", index = False)



The Hateful Eight (2015)

In [153]:
# Uploading the uncleaned CSV file into a new data frame
df_hate_8_kill= pd.read_csv("hateful_eight_kills_uncleaned.csv")

#Adding movie title column
df_hate_8_kill["Movie"] = "The Hateful Eight (2015)"

# Moving 'Movie' column to be the first of the columns
cols = ["Movie"] + [col for col in df_hate_8_kill.columns if col != "Movie"]
df_hate_8_kill =df_hate_8_kill[cols]
#Removing unneccesary rows that have been unnecesarily webscraped
df_hate_8_kill = df_hate_8_kill[:-10]

# Saving to clean csv file, to keep track
df_hate_8_kill.to_csv("hateful_eight_kills_cleaned.csv", index = False)



Once Upon a Time... in Hollywood (2019)

In [154]:
# Uploading the uncleaned CSV file into a new data frame
df_once_upon_kill = pd.read_csv("once_upon_deaths_uncleaned.csv")

# Adding movie title column.
df_once_upon_kill["Movie"]= "Once Upon a Time... in Hollywood (2019)"

# Moving 'Movie' column to be the first of the columns.
cols = ["Movie"] + [col for col in df_once_upon_kill.columns if col != "Movie"]
df_once_upon_kill = df_once_upon_kill[cols]
#Removing unneccesary rows that have been unnecesarily webscraped
df_once_upon_kill= df_once_upon_kill[:-4]
# Saving to clean csv file, to keep track
df_once_upon_kill.to_csv("once_upon_deaths_cleaned.csv", index = False)



Combining all 10 csv files into one data set

In [179]:

#listing all data frames.
kill_dfs = [
    df_res_dog_kill, 
    df_pulp_fiction_kill, 
    df_jackie_brown_kill, 
    df_kill_bill1_kill,
    df_kill_bill2_kill,
    df_death_proof_kill,
    df_ing_bast_kills,
    df_django_unch_kill,
    df_hate_8_kill,
    df_once_upon_kill
]

# joining data frames together
combined_df = pd.concat(kill_dfs, ignore_index=True)

# saving as csv file.
combined_df.to_csv('all_kills.csv', index=False)


# Creating csv for kill count per movie

In [180]:
#using combines_df data frame.
# Count kills per movie (assuming the column is named 'Movie')
kill_counts = combined_df['Movie'].value_counts().reset_index()
#renaming columns.
kill_counts.columns= ['movie', 'kill_count']

# Saving to csv.
kill_counts.to_csv("kill_count_per_movie.csv", index=False)


# Creating csv file categorising kills

In [181]:
#Using the description column to categorise each kill by a type


def categorize_death(desc):
# Convert the description to lowercase to handle potential case differences
    desc= str(desc).lower()

#direct physical combat
    if any(word in desc for word in ['punched', 'beaten', 'bludgeoned', 'kicked', 'fist fight', 'smashed', 'stomped']):
        return 'Physical Combat'

# Weapon-based but blade related
    elif any(word in desc for word in ['stab', 'knife', 'sword', 'cut', 'slashed']):
        return 'Stabbed'

#Weapon-based but gun related.
    elif any(word in desc for word in ['shot', 'gun', 'rifle', 'sniper', 'pistol']):
        return 'Shot'

#explosives
    elif any(word in desc for word in ['exploded', 'grenade', 'bomb', 'mine', 'blown up', 'rocket', 'launcher']):
        return 'Exploded'
#Environmental
    elif any(word in desc for word in ['fell', 'pushed', 'dropped from height', 'crushed', 'run over']):
        return 'Environmental'

# Magic / Supernatural
    elif any(word in desc for word in ['curse', 'spell', 'supernatural', 'hex']):
        return 'Supernatural'

# Poison or indirect
    elif any(word in desc for word in ['poison', 'gas', 'toxin', 'induced', 'drugged']):
        return 'Poisoned'

#Strangulation or suffocation.
    elif any(word in desc for word in ['strangle', 'choked', 'suffocated', 'garroted']):
        return 'Strangled'

# Fire-related
    elif any(word in desc for word in[ 'burn', 'incinerated ']):
        return 'Burned'

#Drowning
    elif 'drown' in desc:
        return 'Drowned'

# Suicide or self carried out
    elif any(word in desc for word in ['suicide', 'killed himself', 'jumped to death', 'sacrificed']):
        return 'Suicide'

#Overdose
    elif any(word in desc for word in ['overdose', 'overdosed']):
        return 'Overdose'

# Hanging
    elif 'hanged' in desc or 'hung' in desc:
        return 'Hanged'

# Ordered or assisted Kill or assasinated.
    elif any(word in desc for word in ['ordered', 'commanded', 'sent someone', 'had someone killed']):
        return 'Assisted Kill'

# Unknown
    else:
        return 'Unknown'

#editing data frame
combined_df['death_type'] = combined_df['Description'].apply(categorize_death)


Creating file that categorises type of death by movie and counts the frequency

In [182]:
# Grouping by film and death_type, then count occurrences.
death_count_per_film_per_type= combined_df.groupby(['Movie', 'death_type']).size().reset_index(name='death_count')
#saving as csv file for record.
death_count_per_film_per_type.to_csv("death_count_per_film_per_type.csv", index=False)

Grouping type of death by occurences across all films

In [183]:
# Group by death_type and count the occurrences across all films
total_death_type_counts= combined_df.groupby('death_type').size().reset_index(name= 'total_death_count')
#saving as csv for record.
total_death_type_counts.to_csv("death_count_per_type.csv", index=False)


# Creating new data frame with IMdB and kills per movie

In [184]:
import pandas as pd

#loading up csv file
kill_count_df = pd.read_csv('kill_count_per_movie.csv')

#Splitting the movie column into 'title' and 'year' so it joins more easily.
kill_count_df[['title', 'year']] = kill_count_df['movie'].str.extract(r'([^\(]+)\s\((\d{4})\)')

# Renaming specific Kill Bill titles to aid the join.
kill_count_df['title'] = kill_count_df['title'].str.strip().replace({
    'Kill Bill: Volume 1': 'Kill Bill: Vol. 1',
    'Kill Bill: Volume 2': 'Kill Bill: Vol. 2'
})

#Dropping the original 'movie' column.
kill_count_df = kill_count_df.drop(columns=['movie'])

# saving to new csv file.
kill_count_df.to_csv('split_movie_title_year.csv', index=False)


In [185]:

#loading up csv file
QT_movies_df = pd.read_csv('Quentin_Tarantino_IMDB_Cleaned.csv')

# Update the title for "Once Upon a Time in Hollywood" as webscrape showed discrpencies.
QT_movies_df.loc[QT_movies_df['Title'] == 'Once Upon a Time in... Hollywood', 'title'] = 'Once Upon a Time... in Hollywood'

#Saving modified file.
QT_movies_df.to_csv('Quentin_Tarantino_IMDB_Cleaned_2.csv', index=False)


In [186]:
import pandas as pd

# Load the CSV file
QT_movies_df = pd.read_csv('Quentin_Tarantino_IMDB_Cleaned.csv')

# Fix the specific title
QT_movies_df['Title'] = QT_movies_df['Title'].replace('Once Upon a Time in... Hollywood', 'Once Upon a Time... in Hollywood')

# Save the corrected CSV
QT_movies_df.to_csv('Quentin_Tarantino_IMDB_Cleaned_Updated.csv', index=False)


In [187]:

#loaidng both csv files.
imdb_df = pd.read_csv('Quentin_Tarantino_IMDB_Cleaned_Updated.csv')

kill_count_df = pd.read_csv('split_movie_title_year.csv')

#merging through 'title'
mix_df = pd.merge(imdb_df, kill_count_df, left_on='Title', right_on='title', how='outer')  
mix_df.head(10)


,IMDb Ranking,Title,Duration (Minutes),Rating,Votes,Genre,Content Rating,URL,Description,Poster,kill_count,title,year
0,2,Reservoir Dogs,99,8.3,1130858,"Crime, Thriller",18,https://www.imdb.com/title/tt0105236/,When a simple jewelry heist goes horribly wron...,https://m.media-amazon.com/images/M/MV5BMmMzYj...,11,Reservoir Dogs,1992
1,3,Pulp Fiction,154,8.9,2330768,"Crime, Drama",18,https://www.imdb.com/title/tt0110912/,"The lives of two mob hitmen, a boxer, a gangst...",https://m.media-amazon.com/images/M/MV5BYTViYT...,15,Pulp Fiction,1994
2,5,Jackie Brown,154,7.5,389331,"Crime, Drama, Thriller",15,https://www.imdb.com/title/tt0119396/,A flight attendant with a criminal past gets n...,https://m.media-amazon.com/images/M/MV5BZmUxZj...,4,Jackie Brown,1997
3,6,Kill Bill: Vol. 1,111,8.2,1248931,"Action, Crime, Thriller",18,https://www.imdb.com/title/tt0266697/,"After waking from a four-year coma, a former a...",https://m.media-amazon.com/images/M/MV5BZmMyYz...,98,Kill Bill: Vol. 1,2003
4,7,Kill Bill: Vol. 2,137,8.0,836763,"Action, Crime, Thriller",18,https://www.imdb.com/title/tt0378194/,The Bride continues her quest of vengeance aga...,https://m.media-amazon.com/images/M/MV5BY2FiNz...,5,Kill Bill: Vol. 2,2004
5,8,Death Proof,127,7.0,324170,"Drama, Thriller",18,https://www.imdb.com/title/tt1028528/,Two separate sets of voluptuous women are stal...,https://m.media-amazon.com/images/M/MV5BYjRlOT...,6,Death Proof,2007
6,9,Inglourious Basterds,153,8.4,1678741,"Adventure, Drama, War",18,https://www.imdb.com/title/tt0361748/,"In Nazi-occupied France during World War II, a...",https://m.media-amazon.com/images/M/MV5BODZhMW...,305,Inglourious Basterds,2009
7,10,Django Unchained,165,8.5,1787746,Western,18,https://www.imdb.com/title/tt1853728/,"With the help of a German bounty-hunter, a fre...",https://m.media-amazon.com/images/M/MV5BMjIyNT...,53,Django Unchained,2012
8,11,The Hateful Eight,168,7.8,690551,"Crime, Drama, Mystery",18,https://www.imdb.com/title/tt3460252/,"In the dead of a Wyoming winter, a bounty hunt...",https://m.media-amazon.com/images/M/MV5BMjA1MT...,74,The Hateful Eight,2015
9,12,Once Upon a Time... in Hollywood,161,7.6,906743,"Comedy, Drama",18,https://www.imdb.com/title/tt7131622/,As Hollywood&apos;s Golden Age is winding down...,https://m.media-amazon.com/images/M/MV5BMzMzNm...,4,Once Upon a Time... in Hollywood,2019


# Creating new csv file detailing the killers

Within the uncleaned data, there is data I want for another csv file focused on kill counts for people actively doing the killing. In some cases, after the webscraping for some of the movies, this piece of information has already been scraped, hence the information will need to webscraped again.

Reservoir Dogs (1992)

In [188]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the Reservoir Dogs death list
url1 = "https://listofdeaths.fandom.com/wiki/Reservoir_Dogs"

#requesting URL
response = requests.get(url1)
soup = BeautifulSoup(response.content, "html.parser")

content = soup.find("div", class_="mw-parser-output")
deaths = []

#creating gender map.
gender_map = {
    "Unnamed Woman": "Female",
    "Unnamed Police Officer": "Male",
    "Unnamed Police Officers": "Male",
    "Officer Marvin Nash": "Male",
    "Officer Frederick \"Freddy\" Newandyke/Mr. Orange": "Male",
    "Victor \"Vic\" Vega/Mr. Blonde": "Male",
    "Joseph \"Joe\" Cabot": "Male",
    "Edward \"Nice Guy Eddie\" Cabot": "Male",
    "Lawrence \"Larry\" Dimmick/Mr. White": "Male",
    "\"Mr. Blue\"": "Male",
    "\"Mr. Brown\"": "Male",
    "\"Mr. Pink\"": "Male",
}

#looping through bullet points.
for li in content.find_all("li"):
    text = li.get_text().strip()

#taking required information
    if ' - ' in text:
        who, desc = text.split(' - ', 1)
        who = who.strip()
        desc =desc.strip()

#dealing with aggregates.
        if who.startswith("Four Unnamed Clerks"):
            for i in range(1, 5):
                deaths.append({
                    "Who Died": f"Unnamed Clerk #{i}",
                    "Description": desc,
                    "Gender": "Male"
                })
        elif who.startswith("Two Unnamed Police Officers"):
            for i in range(1, 3):
                deaths.append({
                    "Who Died": f"Unnamed Police Officer #{i}",
                    "Description": desc,
                    "Gender": "Male"
                })
        else:

            matched_gender= "Unknown"
            for key in gender_map:
                if who.startswith(key) or key in who:
                    matched_gender =gender_map[key]
                    break
            deaths.append({
                "Who Died": who,
                "Description": desc,
                "Gender": matched_gender
            })

#new data fram for killers
df_resdog_killers_uncleaned = pd.DataFrame(deaths)

#Saving
df_resdog_killers_uncleaned.to_csv("reservoir_dogs_killers.csv", index=False)


Now with the uncleaned data for this film specifically, I am going to get the part that discusses the killers.

In [189]:
import pandas as pd

#loading uncleaned csv file to get the killers.
df_resdog_killers_uncleaned= pd.read_csv("reservoir_dogs_killers.csv")

#Adding the Movie and Year columns.
df_resdog_killers_uncleaned["Movie"] = "Reservoir Dogs"
df_resdog_killers_uncleaned["Year"] = 1992

#Moving Movie and Year columns to the front
cols = ["Movie", "Year"] + [col for col in df_resdog_killers_uncleaned.columns if col not in ["Movie", "Year"]]
df_resdog_killers_uncleaned = df_resdog_killers_uncleaned[cols]

#Keeping only the last 4 rows as that is the summary section required
df_resdog_killers_tail = df_resdog_killers_uncleaned.tail(4).reset_index(drop=True)

# Extracting Kill Count and Status from 'Description' before renaming.
df_resdog_killers_tail[['Kills', 'Fate']] = df_resdog_killers_tail['Description'].str.extract(r'(\d+)\s*\((.*?)\)')

#Renaming columns
df_resdog_killers_tail.rename(columns={
    'Who Died': 'Character'
}, inplace=True)

# making sure original Description column is dropped.
df_resdog_killers_tail.drop(columns=['Description'], inplace=True)

#new csv file to keep record
df_resdog_killers_tail.to_csv("reservoir_dogs_killers_cleaned.csv", index=False)


Pulp Fiction (1994)

In [190]:
import pandas as pd

#loading uncleaned csv file to get the killers.
df_pulp_killers = pd.read_csv("pulp_fiction_deaths_uncleaned.csv")  # Make sure your filename matches

#Adding the Movie and Year columns.
df_pulp_killers["Movie"] = "Pulp Fiction"
df_pulp_killers["Year"] =1994

#Moving Movie and Year columns to the front
cols = ["Movie", "Year"] + [col for col in df_pulp_killers.columns if col not in ["Movie", "Year"]]
df_pulp_killers = df_pulp_killers[cols]

gender_map = {
    "Marsellus Wallace": "Male",
    "Jules Winnfield": "Male",
    "Butch Coolidge": "Male",
    "Vincent Vega" : "Male"
}


#Keeping only the last 4 rows as that is the summary section required
df_pulp_killers_tail= df_pulp_killers.tail(4).reset_index(drop=True)

# Extracting Kill Count and Status from 'Description' before renaming.
df_pulp_killers_tail[['Kills', 'Fate']] = df_pulp_killers_tail['Description'].str.extract(r'(\d+)\s*\((.*?)\)')

#Renaming columns
df_pulp_killers_tail.rename(columns={
    'Who Died': 'Character'
}, inplace=True)

df_pulp_killers_tail['Gender']= df_pulp_killers_tail['Character'].map(gender_map)


# making sure original Description column is dropped.
df_pulp_killers_tail.drop(columns=['Description'], inplace=True)

#new csv file to keep record
df_pulp_killers_tail.to_csv("pulp_fiction_killers_cleaned.csv", index=False)



Jackie Brown (1997)

In [191]:
import pandas as pd

#loading uncleaned csv file to get the killers.
df_jackie_killers = pd.read_csv("jackie_brown_deaths_uncleaned.csv")

#Adding the Movie and Year columns.
df_jackie_killers["Movie"] = "Jackie Brown"
df_jackie_killers["Year"] = 1997

#Moving Movie and Year columns to the front
cols = ["Movie", "Year"] + [col for col in df_jackie_killers.columns if col not in ["Movie", "Year"]]
df_jackie_killers= df_jackie_killers[cols]

#Keeping only the last 2 rows as that is the summary section required
df_jackie_killers_tail= df_jackie_killers.tail(2).reset_index(drop=True)

# Extracting Kill Count and Status from 'Description' before renaming.
df_jackie_killers_tail[['Kills', 'Fate']] = df_jackie_killers_tail['Description'].str.extract(r'(\d+)\s*\((.*?)\)')

#Renaming columnsv
df_jackie_killers_tail.rename(columns={
    'Who Died': 'Character'
}, inplace=True)

# making sure original Description column is dropped.
df_jackie_killers_tail.drop(columns=['Description'], inplace=True)

#new csv file to keep record
df_jackie_killers_tail.to_csv("jackie_brown_killers_cleaned.csv", index=False)



Kill Bill Vol 1 and Vol 2 (2003 and 2004 respectively)

In [192]:
import pandas as pd

#loading uncleaned csv file to get the killers.
df_kb1_killers= pd.read_csv("kill_bill1_deaths_uncleaned.csv")
#Adding the Movie and Year columns.
df_kb1_killers["Movie"] = "Kill Bill Vol. 1 and Vol.2"
df_kb1_killers["Year"] = 2003

#Moving Movie and Year columns to the front
cols = ["Movie", "Year"] + [col for col in df_kb1_killers.columns if col not in ["Movie", "Year"]]
df_kb1_killers =df_kb1_killers[cols]

#creating gender map.
gender_map = {
    "Budd/Sidewinder": "Male",
    "Pai Mei": "Male",
    "Elle Driver/California Mountain Snake": "Female",
    "Bill/Snake Charmer": "Male",
    "O-Ren Ishii/Cottonmouth" : "Female",
    "Beatrix Kiddo/The Bride/Black Mamba" : "Female",
    "Vernita Yvonne Green/Copperhead" : "Female",
    "B.B. Kiddo" : "Female",
    "Boss Matsumodo" : "Male",
    "Gogo Yubari" : "Female"
    
}

#Keeping only the last 9 rows as that is the summary section required)
df_kb1_killers_tail = df_kb1_killers.tail(9).reset_index(drop=True)

# Extracting Kill Count and Status from 'Description' before renaming.
df_kb1_killers_tail[['Kills', 'Fate']] = df_kb1_killers_tail['Description'].str.extract(r'(\d+)\s*\((.*?)\)')

#Renaming columns
df_kb1_killers_tail.rename(columns={
    'Who Died': 'Character'
}, inplace=True)

# Apply the gender map to the character column
df_kb1_killers_tail['Gender']= df_kb1_killers_tail['Character'].map(gender_map)

# making sure original Description column is dropped.
df_kb1_killers_tail.drop(columns=['Description'], inplace=True)

#new csv file to keep record
df_kb1_killers_tail.to_csv("kill_bill1_killers_cleaned.csv", index=False)



Death Proof (2007)

In [193]:
import pandas as pd

#loading uncleaned csv file to get the killers.
df_dp_killers = pd.read_csv("death_proof_deaths_uncleaned.csv")

#Adding the Movie and Year columns.
df_dp_killers["Movie"] ="Death Proof"
df_dp_killers["Year"] = 2007

#Moving Movie and Year columns to the front
cols = ["Movie", "Year"] + [col for col in df_dp_killers.columns if col not in ["Movie", "Year"]]
df_dp_killers = df_dp_killers[cols]

#creating gender map.
gender_map = {
    "Abernathy" : "Female",
    "Kim" : "Female",
    "Zoë" : "Female",
    "Stuntman Mike McKay" : "Male"
    
}
#Keeping only the last 4 rows as that is the summary section required
df_dp_killers_tail= df_dp_killers.tail(4).reset_index(drop=True)

# Extracting Kill Count and Status from 'Description' before renaming.
df_dp_killers_tail[['Kills', 'Fate']] = df_dp_killers_tail['Description'].str.extract(r'(\d+)\s*\((.*?)\)')

#Renaming columns
df_dp_killers_tail.rename(columns={
    'Who Died': 'Character'
}, inplace=True)

df_dp_killers_tail['Gender']= df_dp_killers_tail['Character'].map(gender_map)
# making sure original Description column is dropped.
df_dp_killers_tail.drop(columns=['Description'], inplace=True)

#new csv file to keep record
df_dp_killers_tail.to_csv("death_proof_killers_cleaned.csv", index=False)


Inglorious Basterds (2009)

In [194]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

#requesting URL
url7 = "https://listofdeaths.fandom.com/wiki/Inglourious_Basterds"

#Requesting URL
response = requests.get(url7)
soup = BeautifulSoup(response.content, "html.parser")
content = soup.find("div", class_="mw-parser-output")
deaths = []

#creating gender map.
gender_map = {
    "Shosanna Dreyfus": "Female",
    "Fredrick Zoller": "Male",
    "Marcel": "Male",
    "Bridget von Hammersmark": "Female",
    "Hugo Stiglitz": "Male",
    "Donny Donowitz": "Male",
    "Aldo Raine": "Male",
    "Adolf Hitler": "Male",
    "Joseph Goebbels": "Male",
    "Martin Bormann": "Male",
    "Winston Churchill": "Male",
    "Col. Hans Landa": "Male",
    "Lt. Archie Hicox": "Male",
    "Major Dieter Hellstrom": "Male",
    "Sgt. Werner Rachtman": "Male",
    "Pvt. Butz": "Male",
    "Pvt. Willi": "Male",
    "Pfc. Hirschberg": "Male",
    "Pfc. Andy Kagan": "Male",
    "Jakob Dreyfus": "Male",
    "Miriam Dreyfus": "Female",
    "Bob Dreyfus": "Male",
    "Amos Dreyfus": "Male",
    "Eric": "Male",
    "Mathilda": "Female",
    "Francesca Mondino": "Female",
    "Nazi Sergeant Wilhelm": "Male",
    "Wilhelm Wicki": "Male",
    "Omar Ulmer": "Male",
    "Hermann": "Male",
    "Colonel Hans Landa/The Jew Hunter" : "Male"
    
}

# Function to extract number from text
def extract_number(text):
    match = re.search(r'(\d+)', text)
    return int(match.group(1)) if match else 1

#Looping through bullet points.
for li in content.find_all("li"):
    text = li.get_text().strip()

    if ' - ' in text:
        who, desc= text.split(' - ', 1)
        who = who.strip()
        desc= desc.strip()

# Handling aggregates
        aggregate_match= re.match(r'(\d+)\s+(Unnamed|Unnamed Nazi|Unnamed Nazi Soldiers|Unnamed Nazis|Unnamed German Soldiers|Unnamed Members of The Basterds|Unnamed Man|Unnamed Characters)', who)
        if aggregate_match:
            count =extract_number(who)
            base_name = re.sub(r'^\d+\s+', '', who).rstrip('s')
            for i in range(1, count + 1):
                deaths.append({
                    "Who Died": f"{base_name} #{i}",
                    "Description": desc,
                    "Gender": "Male" if "Nazi" in base_name or "Soldier" in base_name else "Unknown"
                })
        else:
            matched_gender = gender_map.get(who, "Unknown")
            deaths.append({
                "Who Died": who,
                "Description": desc,
                "Gender": matched_gender
            })

df_basterds_kills_uncleaned = pd.DataFrame(deaths)

#new csv file to keep record
df_basterds_kills_uncleaned.to_csv("inglourious_basterds_kills_uncleaned.csv", index=False)


In [195]:
import pandas as pd

#loading uncleaned csv file to get the killers.
df_ib_killers = pd.read_csv("inglourious_basterds_kills_uncleaned.csv")

#Adding the Movie and Year columns.
df_ib_killers["Movie"] = "Inglourious Basterds"
df_ib_killers["Year"] = 2009

#Moving Movie and Year columns to the front
cols = ["Movie", "Year"] + [col for col in df_ib_killers.columns if col not in ["Movie", "Year"]]
df_ib_killers = df_ib_killers[cols]

#creating gender map.
gender_map= {
    "Technical Sergeant Hugo Stiglitz": "Male",
    "Corporal Wilhelm Wicki": "Male",
    'Lieutenant Archibald "Archie" Hicox': "Male",
    "First Lieutenant Aldo Raine/The Apache": "Male",
    "Private First Class Gerold Hirschberg": "Male",
    "Private First Class Michael Zimmerman": "Male",
    "Private First Class Simon Sakowitz": "Male",
    "Private First Class Smithson Utivich": "Male",
    "Senior Private Fredrick Zoller": "Male",
    "Private First Class Omar Ulmer": "Male",
    "Staff Sergeant Donny Donowitz/The Bear Jew": "Male",
    "Shosanna Dreyfus": "Female",
    "Fredrick Zoller": "Male",
    "Marcel": "Male",
    "Bridget von Hammersmark": "Female",
    "Hugo Stiglitz": "Male",
    "Donny Donowitz": "Male",
    "Aldo Raine": "Male",
    "Adolf Hitler": "Male",
    "Joseph Goebbels": "Male",
    "Martin Bormann": "Male",
    "Winston Churchill": "Male",
    "Col. Hans Landa": "Male",
    "Lt. Archie Hicox": "Male",
    "Major Dieter Hellstrom": "Male",
    "Sgt. Werner Rachtman": "Male",
    "Pvt. Butz": "Male",
    "Pvt. Willi": "Male",
    "Pfc. Hirschberg": "Male",
    "Pfc. Andy Kagan": "Male",
    "Jakob Dreyfus": "Male",
    "Miriam Dreyfus": "Female",
    "Bob Dreyfus": "Male",
    "Amos Dreyfus": "Male",
    "Eric": "Male",
    "Mathilda": "Female",
    "Francesca Mondino": "Female",
    "Nazi Sergeant Wilhelm": "Male",
    "Wilhelm Wicki": "Male",
    "Omar Ulmer": "Male",
    "Hermann": "Male",
    "Colonel Hans Landa/The Jew Hunter": "Male"
    
}

#Keeping only the last 18 rows as that is the summary section required
df_ib_killers_tail= df_ib_killers.tail(18).reset_index(drop=True)

# Extracting Kill Count and Status from 'Description' before renaming.v
df_ib_killers_tail[['Kills', 'Fate']] = df_ib_killers_tail['Description'].str.extract(r'(\d+)\s*\((.*?)\)')

#Renaming columns
df_ib_killers_tail.rename(columns={
    'Who Died': 'Character'
}, inplace=True)

# Apply the gender map to the 'Character' column
df_ib_killers_tail['Gender'] = df_ib_killers_tail['Character'].map(gender_map)

# making sure original Description column is dropped.
df_ib_killers_tail.drop(columns=['Description'], inplace=True)

#new csv file to keep record
df_ib_killers_tail.to_csv("inglourious_basterds_killers_cleaned.csv", index=False)


Django Unchained (2012)

In [196]:
import pandas as pd

#loading uncleaned csv file to get the killers.
df_django_killers = pd.read_csv("django_unchained_deaths_uncleaned.csv")

#Adding the Movie and Year columns.
df_django_killers["Movie"] = "Django Unchained"
df_django_killers["Year"] = 2012

#Moving Movie and Year columns to the front
cols = ["Movie", "Year"] + [col for col in df_django_killers.columns if col not in ["Movie", "Year"]]
df_django_killers =df_django_killers[cols]

gender_map = {
    "Big Fred":"Male",
    "Jonathan Brittle":"Male",
    "Roger Brittle":"Male",
    "Django Freeman":"Male",
    "Dr. King Schultz":"Male",
    "Calvin J. Candie":"Male",
    "Ellis Brittle":"Male",
    "Smitty Bacal":"Male",
    "Butch Pooch":"Male"

}

#Keeping only the last 4 rows as that is the summary section required
df_django_killers_tail= df_django_killers.tail(9).reset_index(drop=True)

# Extracting Kill Count and Status from 'Description' before renaming.
df_django_killers_tail[['Kills', 'Fate']] = df_django_killers_tail['Description'].str.extract(r'(\d+)\s*\((.*?)\)')

#Renaming columns
df_django_killers_tail.rename(columns={
    'Who Died': 'Character'
}, inplace=True)

df_django_killers_tail['Gender'] = df_django_killers_tail['Character'].map(gender_map)
# making sure original Description column is dropped.
df_django_killers_tail.drop(columns=['Description'], inplace=True)

#new csv file to keep record
df_django_killers_tail.to_csv("django_unchained_killers_cleaned.csv", index=False)


The Hateful Eight (2015)

In [197]:
import pandas as pd

#loading uncleaned csv file to get the killers.
df_hateful8_killers = pd.read_csv("hateful_eight_kills_uncleaned.csv")

#Adding the Movie and Year columns.
df_hateful8_killers["Movie"] = "The Hateful Eight"
df_hateful8_killers["Year"] = 2015
#Moving Movie and Year columns to the front
cols= ["Movie", "Year"] + [col for col in df_hateful8_killers.columns if col not in ["Movie", "Year"]]
df_hateful8_killers = df_hateful8_killers[cols]

#creating gender map.
gender_map = {
    "Major Marquis Warren":"Male",
    "Sheriff Chris Mannix":"Male",
    "Grouch Douglas/Joe Gage":"Male",
    "Jody Domergue":"Male",
    "Pete Hicox/Oswaldo Mobray":"Male",
    "Daisy Domergue": "Female",
    "Lance Lawson": "Male",
    "Marco the Mexican/Señor Bob": "Male"

}

#Keeping only the last 8 rows as that is the summary section required
df_hateful8_killers_tail = df_hateful8_killers.tail(8).reset_index(drop=True)

# Extracting Kill Count and Status from 'Description' before renaming.
df_hateful8_killers_tail[['Kills', 'Fate']]= df_hateful8_killers_tail['Description'].str.extract(r'(\d+)\s*\((.*?)\)')
#Renaming columns
df_hateful8_killers_tail.rename(columns={
    'Who Died': 'Character'
}, inplace=True)

# Apply the gender map to the 'Character' column
df_hateful8_killers_tail['Gender'] = df_hateful8_killers_tail['Character'].map(gender_map)

# making sure original Description column is dropped.
df_hateful8_killers_tail.drop(columns=['Description'], inplace=True)

#new csv file to keep record
df_hateful8_killers_tail.to_csv("hateful_eight_killers_cleaned.csv", index=False)



Once Upon a Time...in Hollywood (2019)

In [198]:
import pandas as pd

#loading uncleaned csv file to get the killers.
df_ouatih_killers = pd.read_csv("once_upon_deaths_uncleaned.csv")

#Adding the Movie and Year columnn
df_ouatih_killers["Movie"] = "Once Upon a Time in Hollywood"
df_ouatih_killers["Year"] =2019

#Moving Movie and Year columns to the front
cols = ["Movie", "Year"] + [col for col in df_ouatih_killers.columns if col not in ["Movie", "Year"]]
df_ouatih_killers= df_ouatih_killers[cols]

gender_map = {
    "Cliff Booth":"Male",
    "Brandy (Dog)":"Male",
    "Grouch Douglas/Joe Gage":"Male",
    "Rick Dalton":"Male"
}


#Keeping only the last 4 rows as that is the summary section required
df_ouatih_killers_tail= df_ouatih_killers.tail(3).reset_index(drop=True)

# Extracting Kill Count and Status from 'Description' before renaming.
df_ouatih_killers_tail[['Kills', 'Fate']] = df_ouatih_killers_tail['Description'].str.extract(r'(\d+)\s*\((.*?)\)')
#Renaming columns
df_ouatih_killers_tail.rename(columns={
    'Who Died': 'Character'
}, inplace=True)
df_ouatih_killers_tail['Gender'] = df_ouatih_killers_tail['Character'].map(gender_map)
# making sure original Description column is dropped.
df_ouatih_killers_tail.drop(columns=['Description'], inplace=True)

#new csv file to keep record
df_ouatih_killers_tail.to_csv("once_upon_killers_cleaned.csv", index=False)


In [199]:
import pandas as pd

# Combining all killer data frames.
killer_df= [
    df_ouatih_killers_tail,
    df_hateful8_killers_tail,
    df_django_killers_tail,
    df_ib_killers_tail,
    df_dp_killers_tail,
    df_kb1_killers_tail,
    df_jackie_killers_tail,
    df_pulp_killers_tail,
    df_resdog_killers_tail
]

#new csv file being creared
all_killers_df = pd.concat(killer_df, ignore_index=True)
all_killers_df.to_csv("all_killers.csv", index=False)



In [200]:
all_killers_df['Kills'] = pd.to_numeric(all_killers_df['Kills'], errors='coerce')

#ordering data into descending order to get top 20 across all films.
all_killers_df_sorted = all_killers_df.sort_values(by='Kills', ascending=False)
all_killers_df_sorted.head(20)
all_killers_df_sorted.head(20).to_csv("top_20_killers.csv", index=False)